# AIMIP catalog — access tour

End-to-end demo of opening AIMIP virtual catalog stores from S3, anonymously, with both access paths and every gotcha you'll hit in practice.

**What you'll see:**
1. How to install and import the right packages
2. Walking the catalog hierarchy
3. **Path A: kerchunk JSON** — stock-tools, no AIMIP install needed
4. **Path B: icechunk** — the only path for ~90 of 239 stores
5. Why some stores have only the icechunk leaf
6. Variables that get dropped from a merged store, and how to find out
7. Per-model peculiarities (DLESyM healpix, UMD-PARETO, ArchesWeather Gen-V2 nested experiments, NVIDIA unstructured)
8. The observations sibling catalog (ERA5, GPCP-SG)

All cells run anonymously against `s3://ai-mip/`. No credentials needed.

## Install

```bash
# minimum: kerchunk JSON path only
pip install xarray intake intake-xarray kerchunk fsspec s3fs zarr jinja2 h5netcdf

# full: also adds the icechunk path (required for ~90 stores)
pip install icechunk aimip-intake-icechunk
```

`jinja2` is required because the catalog YAMLs use `{{ CATALOG_DIR }}` templating but `intake` doesn't pull it automatically. `h5netcdf` is for the observations sibling catalog (raw NetCDFs). `aimip-intake-icechunk` is on PyPI: https://pypi.org/project/aimip-intake-icechunk/.

In [1]:
import warnings
warnings.filterwarnings('ignore')  # numcodecs zarr-v3 + AWS-IMDS warnings are cosmetic

import intake
import xarray as xr

ENDPOINT = 'https://s3.eu-dkrz-1.dkrz.cloud'
AI_CATALOG = 's3://ai-mip/catalog/CMIP6/catalog.yaml'
OBS_CATALOG = 's3://ai-mip/catalog/observations/catalog.yaml'

S3_OPTS = {'anon': True, 'client_kwargs': {'endpoint_url': ENDPOINT}}

## 1. Walk the catalog hierarchy

The AI catalog follows the canonical CMIP6 DRS:

```
cat['AIMIP'][institution_id][source_id][experiment_id][variant_label][table_id][<grid> | <grid>_icechunk]
```

Each leaf is one ensemble member's full set of variables, merged into one `xarray.Dataset`.

In [2]:
cat = intake.open_catalog(AI_CATALOG, storage_options=S3_OPTS)
print('institutions:', list(cat['AIMIP']))
print('NVIDIA models:', list(cat['AIMIP']['NVIDIA']))
print('cBottle experiments:', list(cat['AIMIP']['NVIDIA']['cBottle-1-3']))
print('aimip members:', list(cat['AIMIP']['NVIDIA']['cBottle-1-3']['aimip']))

institutions: ['Ai2', 'ArchesWeather', 'DLESyM', 'Google', 'NVIDIA', 'UMD-PARETO']
NVIDIA models: ['cBottle-1-3']
cBottle experiments: ['aimip', 'aimip-p2k', 'aimip-p4k']
aimip members: ['r1i1p1f1', 'r1i1p2f1', 'r1i1p3f1', 'r1i1p4f1', 'r1i1p5f1']


In [3]:
# Drill all the way down. The leaf-level catalog exposes one or two intake sources per grid:
leaf_node = cat['AIMIP']['NVIDIA']['cBottle-1-3']['aimip']['r1i1p4f1']['Amon']
print('available leaves:', list(leaf_node))

available leaves: ['gn', 'gn_icechunk']


Two leaves: `gn` is the **kerchunk JSON** path (stock zarr; works without `aimip-intake-icechunk`); `gn_icechunk` is the **icechunk** path. They open the same data.

## 2. Path A — kerchunk JSON (stock tools)

No AIMIP-specific install. Reads the JSON sidecar from S3, then fetches chunks lazily.

In [4]:
src = cat['AIMIP']['NVIDIA']['cBottle-1-3']['aimip']['r1i1p4f1']['Amon']['gn']
ds = src.to_dask()
ds

<xarray.Dataset> Size: 6GB
Dimensions:    (time: 555, bnds: 2, i: 49152, plev: 8)
Coordinates:
  * time       (time) datetime64[ns] 4kB 1978-10-16T11:59:59.956800 ... 2024-...
  * i          (i) int32 197kB 0 1 2 3 4 5 ... 49147 49148 49149 49150 49151
    latitude   (i) float64 393kB dask.array<chunksize=(49152,), meta=np.ndarray>
    longitude  (i) float64 393kB dask.array<chunksize=(49152,), meta=np.ndarray>
  * plev       (plev) float64 64B 1e+05 8.5e+04 7e+04 ... 2e+04 5e+03 1e+03
    height     float64 8B ...
Dimensions without coordinates: bnds
Data variables: (12/20)
    time_bnds  (time, bnds) datetime64[ns] 9kB dask.array<chunksize=(555, 2), meta=np.ndarray>
    crs        float32 4B ...
    clivi      (time, i) float32 109MB dask.array<chunksize=(1, 49152), meta=np.ndarray>
    hus        (time, plev, i) float32 873MB dask.array<chunksize=(1, 8, 49152), meta=np.ndarray>
    huss       (time, i) float32 109MB dask.array<chunksize=(1, 49152), meta=np.ndarray>
    pr         (time, i) float32 109MB dask.array<chunksize=(1, 49152), meta=np.ndarray>
    ...         ...
    ts         (time, i) float32 109MB dask.array<chunksize=(1, 49152), meta=np.ndarray>
    ua         (time, plev, i) float32 873MB dask.array<chunksize=(1, 8, 49152), meta=np.ndarray>
    uas        (time, i) float32 109MB dask.array<chunksize=(1, 49152), meta=np.ndarray>
    va         (time, plev, i) float32 873MB dask.array<chunksize=(1, 8, 49152), meta=np.ndarray>
    vas        (time, i) float32 109MB dask.array<chunksize=(1, 49152), meta=np.ndarray>
    zg         (time, plev, i) float32 873MB dask.array<chunksize=(1, 8, 49152), meta=np.ndarray>
Attributes: (12/55)
    Conventions:                 CF-1.7 CMIP-6.2
    activity_id:                 AIMIP
    branch_method:               no parent
    branch_time_in_child:        59400.0
    branch_time_in_parent:       0.0
    contact:                     Python Coder (coder@a.b.c.com)
    ...                          ...
    aimip_dropped_variables:     
    aimip_loaded_variables:      
    aimip_virtualizarr_version:  2.6.0
    aimip_icechunk_version:      2.0.3
    aimip_catalog_version:       0.1.0.dev0
    aimip_store_key:             NVIDIA/cBottle-1-3/aimip/r1i1p4f1/Amon/gn

In [10]:
# Compute one timestep. Chunks fetched on demand from s3://ai-mip/...
tas_t0 = ds['tas'].isel(time=0)
print('global mean tas at t=0:', float(tas_t0.mean().compute()), 'K')

global mean tas at t=0: 287.6286315917969 K


## 3. Path B — icechunk

Same store, alternate driver. Requires `icechunk` and `aimip-intake-icechunk`.

In [11]:
src_ic = cat['AIMIP']['NVIDIA']['cBottle-1-3']['aimip']['r1i1p4f1']['Amon']['gn_icechunk']
ds_ic = src_ic.to_dask()
ds_ic

  2026-04-28T11:45:07.410641Z  WARN aws_config::imds::region: failed to load region from IMDS, err: failed to load IMDS session token: dispatch failure: timeout: HTTP read timeout occurred after 1s: timed out (FailedToLoadToken(FailedToLoadToken { source: DispatchFailure(DispatchFailure { source: ConnectorError { kind: Timeout, source: HttpTimeoutError { kind: "HTTP read", duration: 1s }, connection: Unknown } }) }))
    at /home/conda/feedstock_root/build_artifacts/icechunk_1776344638779/_build_env/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/aws-config-1.8.15/src/imds/region.rs:66



<xarray.Dataset> Size: 6GB
Dimensions:    (time: 555, i: 49152, plev: 8, bnds: 2)
Coordinates:
  * time       (time) datetime64[ns] 4kB 1978-10-16T11:59:59.956800 ... 2024-...
  * i          (i) int32 197kB 0 1 2 3 4 5 ... 49147 49148 49149 49150 49151
    longitude  (i) float64 393kB dask.array<chunksize=(49152,), meta=np.ndarray>
    latitude   (i) float64 393kB dask.array<chunksize=(49152,), meta=np.ndarray>
  * plev       (plev) float64 64B 1e+05 8.5e+04 7e+04 ... 2e+04 5e+03 1e+03
    height     float64 8B ...
Dimensions without coordinates: bnds
Data variables: (12/20)
    crs        float32 4B ...
    prw        (time, i) float32 109MB dask.array<chunksize=(1, 49152), meta=np.ndarray>
    clivi      (time, i) float32 109MB dask.array<chunksize=(1, 49152), meta=np.ndarray>
    huss       (time, i) float32 109MB dask.array<chunksize=(1, 49152), meta=np.ndarray>
    ps         (time, i) float32 109MB dask.array<chunksize=(1, 49152), meta=np.ndarray>
    hus        (time, plev, i) float32 873MB dask.array<chunksize=(1, 8, 49152), meta=np.ndarray>
    ...         ...
    rsut       (time, i) float32 109MB dask.array<chunksize=(1, 49152), meta=np.ndarray>
    rlut       (time, i) float32 109MB dask.array<chunksize=(1, 49152), meta=np.ndarray>
    vas        (time, i) float32 109MB dask.array<chunksize=(1, 49152), meta=np.ndarray>
    zg         (time, plev, i) float32 873MB dask.array<chunksize=(1, 8, 49152), meta=np.ndarray>
    uas        (time, i) float32 109MB dask.array<chunksize=(1, 49152), meta=np.ndarray>
    va         (time, plev, i) float32 873MB dask.array<chunksize=(1, 8, 49152), meta=np.ndarray>
Attributes: (12/55)
    Conventions:                 CF-1.7 CMIP-6.2
    activity_id:                 AIMIP
    branch_method:               no parent
    branch_time_in_child:        59400.0
    branch_time_in_parent:       0.0
    contact:                     Python Coder (coder@a.b.c.com)
    ...                          ...
    aimip_dropped_variables:     
    aimip_loaded_variables:      
    aimip_virtualizarr_version:  2.6.0
    aimip_icechunk_version:      2.0.3
    aimip_catalog_version:       0.1.0.dev0
    aimip_store_key:             NVIDIA/cBottle-1-3/aimip/r1i1p4f1/Amon/gn

In [12]:
# Sanity-check: kerchunk and icechunk return the same numbers
v_kerchunk = float(ds['tas'].isel(time=0, i=0).values)
v_icechunk = float(ds_ic['tas'].isel(time=0, i=0).values)
print(f'kerchunk: {v_kerchunk}')
print(f'icechunk: {v_icechunk}')
assert v_kerchunk == v_icechunk

  2026-04-28T11:45:15.476719Z  WARN aws_config::imds::region: failed to load region from IMDS, err: failed to load IMDS session token: dispatch failure: timeout: client error (Connect): HTTP connect timeout occurred after 1s: timed out (FailedToLoadToken(FailedToLoadToken { source: DispatchFailure(DispatchFailure { source: ConnectorError { kind: Timeout, source: hyper_util::client::legacy::Error(Connect, HttpTimeoutError { kind: "HTTP connect", duration: 1s }), connection: Unknown } }) }))
    at /home/conda/feedstock_root/build_artifacts/icechunk_1776344638779/_build_env/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/aws-config-1.8.15/src/imds/region.rs:66

kerchunk: 260.5030822753906
icechunk: 260.5030822753906


## 4. The icechunk-only stores

Of the 239 AI stores, **90 expose only the `_icechunk` leaf** — there is no plain-grid sibling.

**Why:** these stores went through a build-time workaround for a zarr limitation (ZEP003, variable-length chunks across concatenated files). Some variables are stored as real zarr chunks rather than virtual references. Writing a kerchunk JSON sidecar for those would base64-inline the variable data and force ~32 GiB of RAM during build — so the catalog skips the JSON sidecar for those stores entirely.

**Affected stores:**
- All `Ai2/ACE2-1-ERA5/.../day/...` (60 stores: gn + gr grids × 5 members × 3 experiments)
- All `DLESyM/DLESyM/.../{Amon,day}/gn` (30 stores)
- All `Google/{NeuralGCM,NeuralGCM-HRD}/.../day/gn` (30 stores)

Trying to navigate to the missing `<grid>` leaf returns `KeyError`. Use `<grid>_icechunk` instead.

In [15]:
ai2_day = cat['AIMIP']['Ai2']['ACE2-1-ERA5']['aimip']['r1i1p1f1']['day']
print('available leaves:', list(ai2_day))  # only gn_icechunk + gr_icechunk, no plain gn/gr

available leaves: ['gn_icechunk', 'gr_icechunk']


In [16]:
# Open the icechunk leaf — this is your only option for this group
ds_ai2 = ai2_day['gn_icechunk'].to_dask()
print('data_vars:', sorted(ds_ai2.data_vars))
print('time size:', ds_ai2.sizes['time'])

  2026-04-28T11:46:59.017110Z  WARN aws_config::imds::region: failed to load region from IMDS, err: failed to load IMDS session token: dispatch failure: timeout: client error (Connect): HTTP connect timeout occurred after 1s: timed out (FailedToLoadToken(FailedToLoadToken { source: DispatchFailure(DispatchFailure { source: ConnectorError { kind: Timeout, source: hyper_util::client::legacy::Error(Connect, HttpTimeoutError { kind: "HTTP connect", duration: 1s }), connection: Unknown } }) }))
    at /home/conda/feedstock_root/build_artifacts/icechunk_1776344638779/_build_env/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/aws-config-1.8.15/src/imds/region.rs:66

data_vars: ['hus', 'huss', 'lat_bnds', 'lon_bnds', 'model_layer_bnds', 'pr', 'ps', 'ta', 'tas', 'time_bnds', 'ts', 'ua', 'uas', 'va', 'vas']
time size: 823


In [17]:
# Provenance attr makes the workaround visible:
print('loaded as real zarr (not virtual):')
print(' ', ds_ai2.attrs['aimip_loaded_variables'])

loaded as real zarr (not virtual):
  hus,huss,pr,ps,ta,tas,ts,ua,uas,va,vas


## 5. Variables can be dropped from a merged store

Some providers ship a CMIP6-style group where individual variables disagree on dim sizes (different time coverage, different pressure-level counts, etc.). `xr.merge` cannot reconcile those into one Dataset. The catalog resolves this by picking the majority dim-signature and dropping the minority — recorded in the `aimip_dropped_variables` global attr.

**The metadata's `variables` list shows the source variables**; the actual `ds.data_vars` may exclude some of them. Always cross-check with the provenance attr.

In [18]:
# DLESyM Amon: pr (shorter time) and ta (single plev tied with zg) are dropped
src = cat['AIMIP']['DLESyM']['DLESyM']['aimip']['r1i1p1f1']['Amon']['gn_icechunk']
print('catalog metadata says variables exist:', src.metadata['variables'])
ds = src.to_dask()
print('actually merged into Dataset:       ', sorted(ds.data_vars))
print('dropped (per-store conflict resolution):', ds.attrs['aimip_dropped_variables'])

catalog metadata says variables exist: ['pr', 'ta', 'tas', 'zg']
  2026-04-28T11:47:40.397035Z  WARN aws_config::imds::region: failed to load region from IMDS, err: failed to load IMDS session token: dispatch failure: timeout: client error (Connect): HTTP connect timeout occurred after 1s: timed out (FailedToLoadToken(FailedToLoadToken { source: DispatchFailure(DispatchFailure { source: ConnectorError { kind: Timeout, source: hyper_util::client::legacy::Error(Connect, HttpTimeoutError { kind: "HTTP connect", duration: 1s }), connection: Unknown } }) }))
    at /home/conda/feedstock_root/build_artifacts/icechunk_1776344638779/_build_env/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/aws-config-1.8.15/src/imds/region.rs:66

actually merged into Dataset:        ['tas', 'zg']
dropped (per-store conflict resolution): pr,ta


If you really need a dropped variable, open its raw NetCDF directly via s3fs — the source data is still on S3, just not in this merged store:

```python
import s3fs
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': ENDPOINT}, anon=True)
fs.glob('ai-mip/DLESyM/DLESyM/aimip/r1i1p1f1/Amon/pr/**/*.nc')
```

## 6. Per-model peculiarities — quick tour

### 6a. DLESyM uses a healpix-like grid

Dimensions are `(time, face=12, height=64, width=64)` — no `(lat, lon)`. You'd need a regridder downstream for traditional plotting.

In [19]:
ds_dlesym = cat['AIMIP']['DLESyM']['DLESyM']['aimip']['r1i1p1f1']['Amon']['gn_icechunk'].to_dask()
print('dims:', dict(ds_dlesym.sizes))
print('coords:', list(ds_dlesym.coords))

  2026-04-28T11:48:10.375307Z  WARN aws_config::imds::region: failed to load region from IMDS, err: failed to load IMDS session token: dispatch failure: timeout: client error (Connect): HTTP connect timeout occurred after 1s: timed out (FailedToLoadToken(FailedToLoadToken { source: DispatchFailure(DispatchFailure { source: ConnectorError { kind: Timeout, source: hyper_util::client::legacy::Error(Connect, HttpTimeoutError { kind: "HTTP connect", duration: 1s }), connection: Unknown } }) }))
    at /home/conda/feedstock_root/build_artifacts/icechunk_1776344638779/_build_env/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/aws-config-1.8.15/src/imds/region.rs:66

dims: {'time': 1050, 'face': 12, 'height': 64, 'width': 64, 'plev': 3}
coords: ['face', 'plev', 'time', 'height', 'width']


### 6b. NVIDIA cBottle uses an unstructured `i` grid

`(time, plev, i)` where `i` is a 1D index over healpix-12 cells. `lat`, `lon` are auxiliary coords on `i`.

In [20]:
ds_nvidia = cat['AIMIP']['NVIDIA']['cBottle-1-3']['aimip']['r1i1p4f1']['Amon']['gn'].to_dask()
print('dims:', dict(ds_nvidia.sizes))
print('aux coords on i:', [c for c in ds_nvidia.coords if 'i' in ds_nvidia[c].dims and c != 'i'])

dims: {'time': 555, 'bnds': 2, 'i': 49152, 'plev': 8}
aux coords on i: ['latitude', 'longitude']


### 6c. UMD-PARETO uses regridded output (`gr`) + drops `zg`

UMD-PARETO publishes on a 1° regular grid (`grid_label='gr'`, not `gn`). Every Amon store drops `zg` because it's at a single pressure level (500 hPa, `plev=1`) while `hus`/`ta`/`ua`/`va` are 7-level — incompatible plev sizes. Their experiment names are also normalized: raw paths use `aimip-2k`/`aimip-4k`, the catalog uses `aimip-p2k`/`aimip-p4k`.

In [21]:
ds_umd = cat['AIMIP']['UMD-PARETO']['MD-1p5']['aimip-p4k']['r1i1p1f1']['Amon']['gr_icechunk'].to_dask()
print('grid_label:', ds_umd.attrs.get('grid_label'))
print('plev:', ds_umd['plev'].values)
print('dropped:', ds_umd.attrs['aimip_dropped_variables'])

  2026-04-28T11:48:19.487481Z  WARN aws_config::imds::region: failed to load region from IMDS, err: failed to load IMDS session token: dispatch failure: timeout: client error (Connect): HTTP connect timeout occurred after 1s: timed out (FailedToLoadToken(FailedToLoadToken { source: DispatchFailure(DispatchFailure { source: ConnectorError { kind: Timeout, source: hyper_util::client::legacy::Error(Connect, HttpTimeoutError { kind: "HTTP connect", duration: 1s }), connection: Unknown } }) }))
    at /home/conda/feedstock_root/build_artifacts/icechunk_1776344638779/_build_env/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/aws-config-1.8.15/src/imds/region.rs:66

grid_label: gr
plev: [100000.  85000.  70000.  50000.  25000.  10000.   5000.]
dropped: zg


### 6d. ArchesWeather Gen-V2 has nested experiments

Most providers have one experiment dir per experiment. ArchesWeather Gen-V2 has a nested `aimip-era5/aimip-p2k/` layout — the catalog encodes it as a compound `experiment_id='aimip-era5-p2k'`.

In [22]:
print('AW-Gen-V2 experiments:', list(cat['AIMIP']['ArchesWeather']['ArchesWeatherGen-V2']))

AW-Gen-V2 experiments: ['aimip', 'aimip-era5-p2k', 'aimip-p2k', 'aimip-p4k']


In [23]:
# Open the nested experiment
ds_aw = cat['AIMIP']['ArchesWeather']['ArchesWeatherGen-V2']['aimip-era5-p2k']['r1i1p1f1']['day']['gn_icechunk'].to_dask()
print('data_vars:', sorted(ds_aw.data_vars))
print('dropped:  ', ds_aw.attrs['aimip_dropped_variables'])

  2026-04-28T11:48:24.144717Z  WARN aws_config::imds::region: failed to load region from IMDS, err: failed to load IMDS session token: dispatch failure: timeout: client error (Connect): HTTP connect timeout occurred after 1s: timed out (FailedToLoadToken(FailedToLoadToken { source: DispatchFailure(DispatchFailure { source: ConnectorError { kind: Timeout, source: hyper_util::client::legacy::Error(Connect, HttpTimeoutError { kind: "HTTP connect", duration: 1s }), connection: Unknown } }) }))
    at /home/conda/feedstock_root/build_artifacts/icechunk_1776344638779/_build_env/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/aws-config-1.8.15/src/imds/region.rs:66

data_vars: ['lat_bnds', 'lon_bnds', 'psl', 'tas', 'time_bnds', 'tos', 'uas', 'zg']
dropped:   vas


## 7. Provenance — what to check on every store

Every store carries `aimip_*` global attributes. The two that matter most:

- **`aimip_dropped_variables`** — variables in the source group that were NOT merged into this Dataset (dim-conflict resolver). If non-empty, the catalog is intentionally missing data; open the raw NetCDFs to recover.
- **`aimip_loaded_variables`** — variables stored as real zarr chunks rather than virtual references. Cosmetic for users (same xarray API), useful for diagnosing if a store seems unusually large.

In [24]:
for k in sorted(ds_dlesym.attrs):
    if k.startswith('aimip_'):
        print(f'  {k}: {ds_dlesym.attrs[k]}')

  aimip_build_timestamp: 2026-04-28T10:20:24.220906+00:00
  aimip_catalog_version: 0.1.0.dev0
  aimip_dropped_variables: pr,ta
  aimip_icechunk_version: 2.0.3
  aimip_loaded_variables: ta,tas,zg
  aimip_source_files_count: 7
  aimip_store_key: DLESyM/DLESyM/aimip/r1i1p1f1/Amon/gn
  aimip_virtualizarr_version: 2.6.0


## 8. Observations sibling catalog (ERA5 + GPCP-SG)

Reference data lives in a separate intake catalog. Each entry is a single NetCDF (no merge, no kerchunk/icechunk wrapping). Keys are `<frequency>_<variable_id>`.

In [25]:
obs = intake.open_catalog(OBS_CATALOG, storage_options=S3_OPTS)
print('top:', list(obs))
print('first 8 ERA5 entries:', list(obs['ERA5'])[:8])

top: ['ERA5', 'GPCP-SG']
first 8 ERA5 entries: ['emon_tdps', 'mon_2m_temperature', 'mon_evaporation', 'mon_high_could_cover', 'mon_hus', 'mon_low_cloud_cover', 'mon_mean_sea_level_pressure', 'mon_medium_cloud_cover']


In [26]:
ds_era5 = obs['ERA5']['mon_tas'].to_dask()
ds_era5

<xarray.Dataset> Size: 146MB
Dimensions:               (time: 564, lat: 180, lon: 360, bnds: 2)
Coordinates:
  * time                  (time) datetime64[ns] 5kB 1978-01-17 ... 2024-12-17
  * lat                   (lat) float64 1kB -89.5 -88.5 -87.5 ... 87.5 88.5 89.5
  * lon                   (lon) float64 3kB 0.5 1.5 2.5 ... 357.5 358.5 359.5
    forecast_period       float64 8B ...
    height                float64 8B ...
    originating_centre    |S50 50B ...
Dimensions without coordinates: bnds
Data variables:
    tas                   (time, lat, lon) float32 146MB dask.array<chunksize=(564, 180, 360), meta=np.ndarray>
    latitude_longitude    int32 4B ...
    time_bnds             (time, bnds) datetime64[ns] 9kB dask.array<chunksize=(564, 2), meta=np.ndarray>
    lat_bnds              (lat, bnds) float64 3kB dask.array<chunksize=(180, 2), meta=np.ndarray>
    lon_bnds              (lon, bnds) float64 6kB dask.array<chunksize=(360, 2), meta=np.ndarray>
    forecast_period_bnds  (bnds) float64 16B dask.array<chunksize=(2,), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.7
    software:     Created with ESMValTool v2.14.0.dev68+g5a317f52f.d20260122

## 9. Without intake — direct one-liners

If you don't want to pull in `intake` / `intake-xarray`, the same data is reachable directly. Two patterns:

In [27]:
# === Kerchunk JSON one-liner ===
# Note: target_options *without* asynchronous=True; remote_options *with* asynchronous=True.
# Other combinations either raise 'Loop is not running' (plain Python script)
# or 'Reference-FS's target filesystem must have same value of asynchronous'.
ds = xr.open_dataset(
    'reference://', engine='zarr',
    backend_kwargs={
        'storage_options': {
            'fo': 's3://ai-mip/catalog/NVIDIA/cBottle-1-3/aimip/r1i1p4f1/Amon/gn.json',
            'target_protocol': 's3',
            'target_options': {'anon': True, 'endpoint_url': ENDPOINT},
            'remote_protocol': 's3',
            'remote_options': {'anon': True, 'endpoint_url': ENDPOINT, 'asynchronous': True},
        },
        'consolidated': False,
    },
)
print('opened directly via reference://; vars:', list(ds.data_vars)[:5], '...')

opened directly via reference://; vars: ['time_bnds', 'crs', 'clivi', 'hus', 'huss'] ...


In [28]:
# === Icechunk one-liner ===
# Three details that matter:
#  - force_path_style=True at BOTH layers (DKRZ S3 is MinIO/Ceph-style, virtual-host fails)
#  - authorize_virtual_chunk_access at open() time (not just create())
#  - zarr_format=3, consolidated=False (icechunk is zarr v3, no .zmetadata)
import icechunk

storage = icechunk.s3_storage(
    bucket='ai-mip',
    prefix='catalog/NVIDIA/cBottle-1-3/aimip/r1i1p4f1/Amon/gn',
    endpoint_url=ENDPOINT,
    anonymous=True, force_path_style=True,
)
config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix='s3://ai-mip/',
        store=icechunk.s3_store(
            endpoint_url=ENDPOINT, anonymous=True, s3_compatible=True, force_path_style=True,
        ),
        name='aimip',
    )
)
repo = icechunk.Repository.open(
    storage, config=config, authorize_virtual_chunk_access={'s3://ai-mip/': None},
)
ds = xr.open_zarr(repo.readonly_session('main').store, zarr_format=3, consolidated=False)
print('opened directly via icechunk; vars:', list(ds.data_vars)[:5], '...')

  2026-04-28T11:48:44.633235Z  WARN aws_config::imds::region: failed to load region from IMDS, err: failed to load IMDS session token: dispatch failure: timeout: client error (Connect): HTTP connect timeout occurred after 1s: timed out (FailedToLoadToken(FailedToLoadToken { source: DispatchFailure(DispatchFailure { source: ConnectorError { kind: Timeout, source: hyper_util::client::legacy::Error(Connect, HttpTimeoutError { kind: "HTTP connect", duration: 1s }), connection: Unknown } }) }))
    at /home/conda/feedstock_root/build_artifacts/icechunk_1776344638779/_build_env/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/aws-config-1.8.15/src/imds/region.rs:66

opened directly via icechunk; vars: ['clivi', 'crs', 'huss', 'hus', 'pr'] ...


## 10. Summary

| Group | Leaves available | Why |
|---|---|---|
| NVIDIA cBottle Amon, day | both `<grid>` + `<grid>_icechunk` | normal case |
| ArchesWeather V2 / Gen-V2 (most) | both | normal case |
| Google NeuralGCM Amon | both | normal case |
| UMD-PARETO MD-1p5 Amon | both (`gr` + `gr_icechunk`) | normal case; drops `zg` |
| Ai2 ACE2-1-ERA5 day | only `<grid>_icechunk` | ZEP003 retry path; JSON sidecar skipped |
| DLESyM Amon, day | only `<grid>_icechunk` | ZEP003 retry path; drops `pr,ta` |
| Google NeuralGCM(-HRD) day | only `<grid>_icechunk` | ZEP003 retry path |
| ArchesWeather Gen-V2 `aimip-era5-p2k` day | only `<grid>_icechunk` | ZEP003 retry path; drops `vas` |

**Decision rule:** if you only install `kerchunk` + `intake-xarray`, you can open the 149 stores with both leaves. To open the remaining 90, install `icechunk` + `aimip-intake-icechunk`. The icechunk leaf is also a perfectly fine default choice for ALL stores — it's faster and more robust than kerchunk JSON.

Always check `ds.attrs['aimip_dropped_variables']` before relying on a Dataset to be complete relative to its `metadata['variables']` list.